# Oil Data Preprocessing Pipeline

This notebook converts raw oil market data from multiple sources into clean, analysis-ready datasets.

## Overview
- **Input**: Excel file with 5 sheets (daily, weekly, monthly data + macro indicators)
- **Output**: 3 CSV files ready for forecasting
  - `Oil_Clean_Weekly.csv` - Full cleaned dataset
  - `Oil_Clean_Weekly_Stn.csv` - Stationarity-transformed version
  - `Oil_Model_Weekly_Core.csv` - Model-ready features with lags

## Key Transformations
- Align all data to weekly frequency (Friday)
- Compute log returns for price series
- Handle missing data systematically
- Create lagged features for time-series modeling

In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

## Configuration

Set up file paths and processing parameters.

In [7]:
# File paths
DATA_XLSX = Path("../Data/Oil Data.xlsx")
OUT_CLEAN = Path("../Data/Oil_Clean_Weekly.csv")
OUT_STN   = Path("../Data/Oil_Clean_Weekly_Stn.csv")
OUT_MODEL = Path("../Data/Oil_Model_Weekly_Core.csv")

# Processing parameters
STATIONARITY_MAX_MISS = 0.10  # Drop columns with >10% missing after transformation
Z_SCORE_STN = True            # Standardize stationarity-ready predictors
MODEL_LAGS = 4                # Number of AR lags for target variable
MODEL_EXOG_LAG = 1            # Lag exogenous features by 1 week to avoid look-ahead bias
STANDARDIZE_MODEL_PRED = False # Optional: standardize model predictors

## Helper Functions

Utility functions for data cleaning and transformation.

In [8]:
def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Convert column names to snake_case and handle special characters."""
    return df.rename(columns=lambda c: "_".join(
        str(c).strip().lower()
        .replace("%", "pct")
        .replace("&", "and")
        .replace("/", "_")
        .replace("-", "_")
        .split()
    ))

def ensure_datetime_index(df: pd.DataFrame) -> pd.DataFrame:
    """Identify date column and set as sorted datetime index."""
    d = df.copy()
    date_col = None
    for c in d.columns:
        if "date" in str(c).lower():
            date_col = c
            break
    if date_col is None:
        date_col = d.columns[0]
    d[date_col] = pd.to_datetime(d[date_col])
    return d.set_index(date_col).sort_index()

def resample_weekly_last(df: pd.DataFrame) -> pd.DataFrame:
    """Resample to weekly frequency (Friday), taking the last value."""
    return df.resample("W-FRI").last()

def resample_weekly_ffill(df: pd.DataFrame) -> pd.DataFrame:
    """Resample to weekly frequency (Friday), forward-filling values."""
    return df.resample("W-FRI").ffill()

def log_return(series: pd.Series, periods: int = 1) -> pd.Series:
    """Calculate log returns: log(price_t / price_t-1)."""
    return np.log(series / series.shift(periods))

def is_return_like(name: str) -> bool:
    """Check if column name indicates it's already a return series."""
    n = name.lower()
    return ("_ret" in n) or (n.endswith("returns")) or (n == "returns") or (n == "dxy_returns")

def transform_levels_to_stationary(s: pd.Series) -> pd.Series:
    """Transform price levels to stationary series using log-diff or first-diff."""
    s = s.astype(float)
    if np.isfinite(s).sum() < 3:
        return pd.Series(index=s.index, dtype=float)
    # Log-diff for positive series, first-diff otherwise
    if (s > 0).all():
        return np.log(s).diff()
    else:
        return s.diff()

## Step 1: Load and Normalize Data

Load all 5 sheets from the Excel file and:
- Standardize column names to snake_case
- Convert dates to datetime index
- Align all data to weekly frequency (Friday)
- Compute returns for price series

In [9]:
def load_all_sheets(xlsx_path: Path):
    """Load and normalize all 5 data sheets from Excel file."""
    xls = pd.ExcelFile(xlsx_path)

    # 1) DAILY DATA -> resample to weekly (take Friday close)
    ind_d = pd.read_excel(xls, sheet_name="Industry_D")
    ind_d = standardize_columns(ind_d)
    ind_d = ensure_datetime_index(ind_d)
    ind_d_w = resample_weekly_last(ind_d)
    
    # Compute returns for crude oil and related commodities
    for col in ["cl1", "brent", "cl2", "crkss12_index", "bdiy_index"]:
        if col in ind_d_w.columns:
            ind_d_w[f"{col}_ret"] = log_return(ind_d_w[col])

    # 2) WEEKLY DATA -> keep as-is (already weekly frequency)
    ind_w = pd.read_excel(xls, sheet_name="Industry_W")
    ind_w = standardize_columns(ind_w)
    ind_w = ensure_datetime_index(ind_w)

    # 3) MONTHLY INDUSTRY DATA -> resample to weekly (forward-fill)
    ind_m = pd.read_excel(xls, sheet_name="Industry_M")
    ind_m = standardize_columns(ind_m)
    ind_m = ensure_datetime_index(ind_m)
    ind_m_w = resample_weekly_ffill(ind_m)
    # Drop price columns already in daily data to avoid duplicates
    for c in ["cl1", "cl2", "brent", "crks321c_index", "baltic_dry_index"]:
        if c in ind_m_w.columns:
            ind_m_w = ind_m_w.drop(columns=c)

    # 4) MONTHLY MACRO DATA -> resample to weekly (forward-fill)
    macro_m = pd.read_excel(xls, sheet_name="Macro_M")
    macro_m = standardize_columns(macro_m)
    macro_m = ensure_datetime_index(macro_m)
    macro_w = resample_weekly_ffill(macro_m)
    # Create term spread if both yields exist
    if "3m_yield" in macro_w.columns and "10y_yield" in macro_w.columns:
        macro_w["term_spread"] = macro_w["10y_yield"] - macro_w["3m_yield"]
    # Compute SPX returns if level exists
    if "spx" in macro_w.columns and "spx_ret" not in macro_w.columns:
        macro_w["spx_ret"] = log_return(macro_w["spx"])

    # 5) MONTHLY STOCK PRICES -> resample to weekly (forward-fill) and compute returns
    stocks_m = pd.read_excel(xls, sheet_name="StockPrices_M")
    stocks_m = standardize_columns(stocks_m)
    stocks_m = ensure_datetime_index(stocks_m)
    stocks_w = resample_weekly_ffill(stocks_m)
    # Compute returns for all numeric columns
    for col in stocks_w.columns:
        if stocks_w[col].dtype.kind in "biufc" and not col.endswith("_ret"):
            stocks_w[f"{col}_ret"] = log_return(stocks_w[col])

    return ind_d_w, ind_w, ind_m_w, macro_w, stocks_w

## Step 2: Build Clean Weekly Panel

Merge all sheets into a single weekly panel and:
- Handle missing data (forward-fill small gaps, drop columns with >30% missing)
- Ensure target variable (cl1_ret) is present
- Create dense panel by removing rows with any NaN

In [10]:
def build_clean_weekly(ind_d_w, ind_w, ind_m_w, macro_w, stocks_w) -> pd.DataFrame:
    """Merge all weekly data into one clean panel."""
    # Start with daily->weekly as base (has oil prices)
    master = ind_d_w.copy()

    # Merge numeric columns from other sheets
    for df_to_merge in [ind_w, ind_m_w, macro_w, stocks_w]:
        num_cols = df_to_merge.select_dtypes(include=[np.number]).columns.tolist()
        master = master.join(df_to_merge[num_cols], how="left")

    # Ensure target variable exists
    if "cl1_ret" not in master.columns and "cl1" in master.columns:
        master["cl1_ret"] = log_return(master["cl1"])

    # Remove NaN from return calculation
    master = master.dropna(subset=["cl1_ret"])

    # Missing data strategy
    miss_ratio = master.isna().mean()

    # Forward/backward fill small gaps (< 5% missing)
    fill_cols = miss_ratio[(miss_ratio > 0) & (miss_ratio < 0.05)].index.tolist()
    if fill_cols:
        master[fill_cols] = master[fill_cols].ffill().bfill()

    # Drop columns with heavy missingness (> 30%)
    drop_cols = master.columns[master.isna().mean() > 0.30].tolist()
    if drop_cols:
        master = master.drop(columns=drop_cols)

    # Create dense panel (no missing values)
    master_clean = master.dropna()

    return master_clean, drop_cols

## Step 3: Build Stationarity-Ready Panel

Transform data for statistical modeling:
- Keep return-like series as-is (already differenced)
- Convert price levels to differences (log-diff for positive, first-diff otherwise)
- Standardize all predictors using z-score
- Drop predictors with >10% missing after transformation

In [11]:
def build_stationarity_ready(clean_df: pd.DataFrame) -> pd.DataFrame:
    """Transform predictors to be stationary for regression analysis."""
    if "cl1_ret" not in clean_df.columns:
        raise ValueError("cl1_ret missing in clean weekly panel.")

    numeric_cols = clean_df.select_dtypes(include=[np.number]).columns.tolist()
    target = "cl1_ret"
    predictors = [c for c in numeric_cols if c != target]

    out = pd.DataFrame(index=clean_df.index)
    out[target] = clean_df[target]

    kept, dropped = [], []

    for col in predictors:
        s = clean_df[col]
        
        # Keep returns as-is, transform levels
        if is_return_like(col):
            t = s.copy()
        else:
            t = transform_levels_to_stationary(s)

        # Filter out predictors with too much missing data
        if t.isna().mean() > STATIONARITY_MAX_MISS:
            dropped.append(col)
            continue

        # Standardize using z-score (mean=0, std=1)
        if Z_SCORE_STN:
            sd = t.std(ddof=0)
            if np.isfinite(sd) and sd > 0:
                t = (t - t.mean()) / sd
            else:
                dropped.append(col)
                continue

        out[col] = t
        kept.append(col)

    # Remove any remaining missing values
    out = out.dropna()
    return out

## Step 4: Build Model-Ready Dataset

Create features optimized for forecasting:
- **Target**: cl1_ret (crude oil returns)
- **AR Lags**: 4 lagged values of target
- **Cross-asset returns**: Brent, WTI futures, equity indices
- **Industry features**: Production, rigs, stocks, open interest (as differences)
- **Macro features**: Interest rates, VIX (as differences)
- **Time alignment**: Shift exogenous variables by 1 week to avoid look-ahead bias

In [12]:
def build_model_core(clean_df: pd.DataFrame) -> pd.DataFrame:
    """Build feature set for time-series forecasting model."""
    if "cl1_ret" not in clean_df.columns:
        raise ValueError("cl1_ret missing in clean weekly panel.")

    model = pd.DataFrame(index=clean_df.index)
    
    # TARGET: CL1 returns
    model["cl1_ret"] = clean_df["cl1_ret"]

    # AR LAGS: Previous 4 weeks of returns
    for L in range(1, MODEL_LAGS + 1):
        model[f"cl1_ret_l{L}"] = clean_df["cl1_ret"].shift(L)

    # CROSS-ASSET RETURNS: Related commodities and equity indices
    cross_ret = ["brent_ret", "cl2_ret", "bdiy_index_ret", "spx_ret", "dxy_returns"]
    for col in cross_ret:
        if col in clean_df.columns:
            model[col] = clean_df[col]

    # INDUSTRY LEVELS -> DIFFERENCES: Production, rigs, crude stocks, open interest
    weekly_levels = [
        "weekly_prod_crude", "weekly_rig_count", "weekly_change_in_crude_stock",
        "weekly_commercial_long", "weekly_commercial_short", "weekly_total_open_interest"
    ]
    for col in weekly_levels:
        if col in clean_df.columns:
            model[col + "_diff"] = clean_df[col].diff()

    # MACRO LEVELS -> DIFFERENCES: Yield spreads, volatility
    macro_levels = ["3m_yield", "10y_yield", "term_spread", "vix"]
    for col in macro_levels:
        if col in clean_df.columns:
            model[col + "_diff"] = clean_df[col].diff()

    # OIL COMPANY EQUITY RETURNS: Sentiment from major oil firms
    oil_eq_ret = [
        "dow_jones_us_oilandgas_index_ret", "exxonmobil_ret", "conocophilips_ret",
        "chevron_ret", "bp_ret", "totalenergies_ret"
    ]
    for col in oil_eq_ret:
        if col in clean_df.columns:
            model[col] = clean_df[col]

    # AVOID LOOK-AHEAD BIAS: Shift exogenous features forward by 1 week
    # (they're typically released after the period ends)
    exog_cols = [
        c for c in model.columns
        if c != "cl1_ret" and not c.startswith("cl1_ret_l")
    ]
    if MODEL_EXOG_LAG > 0:
        model[exog_cols] = model[exog_cols].shift(MODEL_EXOG_LAG)

    # Remove NaN from lags, differences, and shifts
    model = model.dropna()

    # OPTIONAL: Standardize predictors (not the target)
    if STANDARDIZE_MODEL_PRED:
        pred_cols = [c for c in model.columns if c != "cl1_ret"]
        model_mean = model[pred_cols].mean()
        model_std = model[pred_cols].std(ddof=0).replace(0, 1)
        model[pred_cols] = (model[pred_cols] - model_mean) / model_std

    return model

## Step 5: Run the Full Pipeline

Execute all processing steps and generate output files.

In [13]:
# STEP 1: Load all data
print("Loading data from Excel...")
ind_d_w, ind_w, ind_m_w, macro_w, stocks_w = load_all_sheets(DATA_XLSX)
print("✓ Loaded 5 sheets successfully\n")

# STEP 2: Build clean weekly panel
print("Building clean weekly panel...")
clean_df, dropped_cols = build_clean_weekly(ind_d_w, ind_w, ind_m_w, macro_w, stocks_w)
clean_df.to_csv(OUT_CLEAN)

print("Clean weekly panel:")
print(f"  Rows: {len(clean_df):,}  |  Cols: {clean_df.shape[1]}")
print(f"  Date range: {clean_df.index.min().date()} → {clean_df.index.max().date()}")
if dropped_cols:
    print(f"  Dropped {len(dropped_cols)} columns (>30% missing)")
print(f"  ✓ Saved: {OUT_CLEAN}\n")

# STEP 3: Build stationarity-ready panel
print("Building stationarity-ready panel...")
stn_df = build_stationarity_ready(clean_df)
stn_df.to_csv(OUT_STN)

print("Stationarity-Ready panel:")
print(f"  Rows: {len(stn_df):,}  |  Cols: {stn_df.shape[1]}")
print(f"  Date range: {stn_df.index.min().date()} → {stn_df.index.max().date()}")
print(f"  ✓ Saved: {OUT_STN}\n")

# STEP 4: Build model-ready dataset
print("Building model-ready dataset...")
model_df = build_model_core(clean_df)
model_df.to_csv(OUT_MODEL)

print("Model-ready weekly dataset:")
print(f"  Rows: {len(model_df):,}  |  Cols: {model_df.shape[1]}")
print(f"  Date range: {model_df.index.min().date()} → {model_df.index.max().date()}")
print(f"  ✓ Saved: {OUT_MODEL}\n")

print("=" * 60)
print("Pipeline complete! All datasets generated successfully.")
print("=" * 60)

Loading data from Excel...
✓ Loaded 5 sheets successfully

Building clean weekly panel...
Clean weekly panel:
  Rows: 1,538  |  Cols: 44
  Date range: 1995-01-13 → 2024-06-28
  Dropped 2 columns (>30% missing)
  ✓ Saved: ../Data/Oil_Clean_Weekly.csv

Building stationarity-ready panel...
Stationarity-Ready panel:
  Rows: 1,537  |  Cols: 44
  Date range: 1995-01-20 → 2024-06-28
  ✓ Saved: ../Data/Oil_Clean_Weekly_Stn.csv

Building model-ready dataset...
Model-ready weekly dataset:
  Rows: 1,534  |  Cols: 26
  Date range: 1995-02-10 → 2024-06-28
  ✓ Saved: ../Data/Oil_Model_Weekly_Core.csv

Pipeline complete! All datasets generated successfully.


In [15]:
path = "../Data/Oil_Model_Weekly_Core.csv"
df = pd.read_csv(path)
df.head()

,date,cl1_ret,cl1_ret_l1,cl1_ret_l2,cl1_ret_l3,cl1_ret_l4,brent_ret,cl2_ret,bdiy_index_ret,spx_ret,...,3m_yield_diff,10y_yield_diff,term_spread_diff,vix_diff,dow_jones_us_oilandgas_index_ret,exxonmobil_ret,conocophilips_ret,chevron_ret,bp_ret,totalenergies_ret
0,1995-02-10,-0.017186,0.045202,-0.038256,0.062503,-0.008525,-0.000615,0.033584,-0.016824,0.023988,...,-0.057,-0.380,-0.323,-1.24,0.007990,0.028399,-0.027080,0.000000,-0.028578,-0.038881
1,1995-02-17,0.024085,-0.017186,0.045202,-0.038256,0.062503,0.022514,-0.009247,-0.016097,0.000000,...,0.000,0.000,0.000,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,1995-02-24,-0.011702,0.024085,-0.017186,0.045202,-0.038256,0.017298,0.023225,-0.002539,0.000000,...,0.000,0.000,0.000,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,1995-03-03,-0.003215,-0.011702,0.024085,-0.017186,0.045202,-0.031234,-0.012896,0.026093,0.000000,...,0.000,0.000,0.000,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,1995-03-10,-0.039414,-0.003215,-0.011702,0.024085,-0.017186,-0.013514,-0.003793,0.025429,0.035439,...,-0.068,-0.005,0.063,-0.21,0.041418,0.021765,0.045982,0.065064,-0.014596,-0.022269
